# EnbPI: explore the reproduction

This notebook reads saved results; it does not train models. Run `scripts/run_experiment.py --config configs/reproduction.json` first. The experiment is the lag-only random-forest portion of Figure 1 in Xu and Xie (ICML 2021).

The original experiment uses solar irradiance. Retail demand is the next application, not a result already established here.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/reproduction.json').exists())
results = root / 'outputs/solar_reproduction'
assert (results / 'summary.csv').exists(), 'Run the reproduction first.'
print('Project:', root)

## 1. Check what was run

The 20 lag features use only preceding values. The 10 seeds repeat randomized training on the same time period.

In [ ]:
metadata = json.loads((results / 'run_metadata.json').read_text())
for key in ['rows', 'train_raw', 'train_effective', 'test_rows', 'test_start', 'test_end', 'upstream_commit']:
    print(f'{key}: {metadata[key]}')

## 2. Compare with the authors

Coverage is the fraction of observations within the interval. Compare width as well. Reference values come from the authors' saved CSV.

In [ ]:
comparison = pd.read_csv(results / 'upstream_comparison.csv')
columns = ['target_coverage', 'coverage_mean', 'upstream_coverage', 'coverage_difference_pp', 'width_mean', 'upstream_width']
print(comparison[columns].round(4).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.imshow(plt.imread(results / 'coverage_width.png'))
ax.axis('off')
plt.show()

## 3. Inspect one seed at a 90% target

Each interval is issued before its actual value is used to update the error window. A 90% nominal target is not a guarantee for each outcome.

In [ ]:
predictions = pd.read_csv(results / 'predictions/seed_98765_alpha_0.10.csv.gz', parse_dates=['timestamp'])
covered = (predictions.actual >= predictions.lower) & (predictions.actual <= predictions.upper)
print(f'Observed coverage: {covered.mean():.2%}')
print(f'Mean width: {(predictions.upper - predictions.lower).mean():.2f} W/m²')
print(predictions.head().to_string(index=False))

In [ ]:
week = predictions.iloc[:168]
fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(week.timestamp, week.lower, week.upper, alpha=0.25, label='90% nominal interval')
ax.plot(week.timestamp, week.actual, color='black', label='Actual')
ax.plot(week.timestamp, week.center, label='Forecast center')
ax.set(ylabel='DHI (W/m²)', title='First seven test days')
ax.legend()
fig.autofmt_xdate()
plt.show()

## 4. Look beyond the overall average

Overall 90% coverage can hide systematic misses at particular hours. This table uses the first seed; the report also aggregates all ten seeds.

In [ ]:
hourly = predictions.assign(hour=predictions.timestamp.dt.hour, hit=covered).groupby('hour').agg(coverage=('hit', 'mean'), observations=('hit', 'size'))
print(hourly.round(3).to_string())
ax = hourly.coverage.plot.bar(figsize=(12, 3), color='#0f766e')
ax.axhline(0.9, color='#b45309', linestyle='--')
ax.set(ylabel='Coverage', title='Coverage by local hour: first seed', ylim=(0, 1.05))
plt.tight_layout()
plt.show()

## 5. Your interpretation

Write your answers here before presenting this project:

1. How close are the local results to the authors' results?
2. At which hours is coverage weakest?
3. Why is the seed standard deviation not a confidence interval for a new year?
4. What would change when forecasting daily retail sales?
5. Which parts reproduce the paper and which are additional diagnostics?

Sources: [paper](https://proceedings.mlr.press/v139/xu21h.html), [author code](https://github.com/hamrel-cxu/EnbPI). See `docs/reproduction-notes.md` for differences.